In [1]:
import pandas as pd
import sqlite3
import logging
from ingestion_db import ingest_db

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)


def create_vendor_summary(conn):
    """
    This function merges different tables
    and creates vendor sales summary.
    """

    query = """
    SELECT

        pp.[Item ID],
        pp.[Item Name],
        pp.[Category],
        vi.[Vendor Name] AS VendorName,
        pp.[Supplier],

        pp.[Avg Purchase Price ($)] AS AvgPurchasePrice,
        pp.[Net Price ($)] AS NetPurchasePrice,

        SUM(s.[Quantity Sold]) AS TotalSalesQuantity,
        SUM(s.[Total Sale ($)]) AS TotalSalesAmount,
        SUM(s.[Net Revenue ($)]) AS TotalNetRevenue,
        SUM(s.[Gross Profit ($)]) AS TotalGrossProfit,

        SUM(vi.[Quantity]) AS TotalPurchaseQuantity,
        SUM(vi.[Total Amount ($)]) AS TotalPurchaseAmount,
        SUM(vi.[Amount Paid ($)]) AS TotalAmountPaid,
        SUM(vi.[Balance Due ($)]) AS TotalBalanceDue

    FROM purchase_prices pp

    JOIN sales s
        ON pp.[Item ID] = s.[Item ID]

    JOIN vendor_invoice vi
        ON pp.[Item ID] = vi.[Item ID]

    GROUP BY
        pp.[Item ID],
        pp.[Item Name],
        pp.[Category],
        vi.[Vendor Name],
        pp.[Supplier],
        pp.[Avg Purchase Price ($)],
        pp.[Net Price ($)]

    ORDER BY TotalSalesAmount DESC
    """

    vendor_sales_summary = pd.read_sql_query(query, conn)

    return vendor_sales_summary


def clean_data(df):
    """
    This function cleans data.
    """

    # datatype conversion
    df["NetPurchasePrice"] = df["NetPurchasePrice"].astype("float64")

    # fill null values
    df.fillna(0, inplace=True)

    # remove extra spaces
    df["VendorName"] = df["VendorName"].astype(str).str.strip()

    # calculated columns
    df["TotalGrossProfit"] = (
        df["TotalSalesAmount"]
        - df["TotalPurchaseAmount"]
    )

    df["TotalBalanceDuePercent"] = (
        df["TotalGrossProfit"]
        / df["TotalSalesAmount"]
    ) * 100

    df["StockTurnover"] = (
        df["TotalSalesQuantity"]
        / df["TotalPurchaseQuantity"]
    )

    df["SalestoPurchaseRatio"] = (
        df["TotalSalesAmount"]
        / df["TotalPurchaseAmount"]
    )

    return df


if __name__ == "__main__":
    conn = sqlite3.connect("inventory.db")

    logging.info("Creating Vendor Summary Table....")
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info("Cleaning Data.....")
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info("Ingesting Data.....")
    ingest_db(clean_df, "vendor_sales_summary", conn)

    logging.info("Completed")

    conn.close()

In [2]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', 'pd', 'sqlite3', 'logging', 'ingest_db', 'create_vendor_summary', 'clean_data', 'conn', 'summary_df', 'clean_df', '_i2'])


In [5]:
get_vendor_summary.to_csv("get_vendor_summary.csv", index=False)

NameError: name 'get_vendor_summary' is not defined

In [6]:
for name in globals():
    if "vendor" in name.lower():
        print(name)

RuntimeError: dictionary changed size during iteration

In [7]:
for name in globals():
    if "summary" in name.lower():
        print(name)

create_vendor_summary
summary_df
